In [1]:
# ============================================================
# 1. Imports & Device
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


PyTorch version: 2.13.0+cu126
Using device: cpu


In [2]:
# ============================================================
# 2. CIFAR-10 Dataset & DataLoaders
#    CIFAR-10: 32×32 color images, 10 classes
# ============================================================

# Transform: convert to tensor + normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    # Normalize each channel: mean & std for CIFAR-10
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

# Download training & test sets
train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# DataLoaders (you can adjust batch_size)
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True, num_workers=2)

test_loader = DataLoader(test_dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

len(train_dataset), len(test_dataset)


(50000, 10000)

In [3]:
# ============================================================
# 3. Define CNN (matches your lecture architecture)
#    Input:  3×32×32
#    Conv1:  3 → 16 filters, 5×5, no padding
#      Output size: 16×28×28
#    Pool1: 2×2 → 16×14×14
#    Conv2: 16 → 32 filters, 5×5
#      Output size: 32×10×10
#    Pool2: 2×2 → 32×5×5
#    FC:    32*5*5 → 10 classes
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Conv layer 1: (3, 32, 32) -> (16, 28, 28)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16,
                               kernel_size=5, stride=1, padding=0)
        # Conv layer 2: (16, 14, 14) -> (32, 10, 10)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32,
                               kernel_size=5, stride=1, padding=0)
        # Max pooling (2×2) after each conv
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layer:
        # After two pools, image 32×32 → 5×5×32 feature map:
        # 32×32 --Conv1(5x5)--> 28×28 --Pool--> 14×14
        # 14×14 --Conv2(5x5)--> 10×10 --Pool--> 5×5
        self.fc = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        # Conv1 + ReLU + Pool
        x = self.pool(torch.relu(self.conv1(x)))   # (batch, 16, 14, 14)
        # Conv2 + ReLU + Pool
        x = self.pool(torch.relu(self.conv2(x)))   # (batch, 32, 5, 5)
        # Flatten
        x = x.view(x.size(0), -1)                  # (batch, 32*5*5)
        # Fully connected → logits for 10 classes
        x = self.fc(x)
        return x

model = SimpleCNN().to(device)
print(model)


SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=800, out_features=10, bias=True)
)


In [4]:
# ============================================================
# 4. Loss Function & Optimizer
# ============================================================

criterion = nn.CrossEntropyLoss()        # combines LogSoftmax + NLLLoss
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# For demo: only a few epochs so it runs quickly in class
num_epochs = 3


In [5]:
# ============================================================
# 5. Training Loop
# ============================================================

def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Zero gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(inputs)

        # 3. Compute loss
        loss = criterion(outputs, labels)

        # 4. Backprop
        loss.backward()

        # 5. Update weights
        optimizer.step()

        # Stats
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        running_correct += predicted.eq(labels).sum().item()

        # Print a mini status every ~200 batches
        if (batch_idx + 1) % 200 == 0:
            print(f"  [Batch {batch_idx+1:3d}] Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    print(f"Epoch {epoch+1} Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc*100:.2f}%")
    return epoch_loss, epoch_acc

for epoch in range(num_epochs):
    print(f"===== Epoch {epoch+1}/{num_epochs} =====")
    train_one_epoch(epoch)


===== Epoch 1/3 =====
  [Batch 200] Loss: 1.7583
  [Batch 400] Loss: 1.3303
  [Batch 600] Loss: 1.0756
Epoch 1 Train Loss: 1.4554 | Train Acc: 47.85%
===== Epoch 2/3 =====
  [Batch 200] Loss: 1.3156
  [Batch 400] Loss: 1.4963
  [Batch 600] Loss: 0.9066
Epoch 2 Train Loss: 1.1255 | Train Acc: 60.79%
===== Epoch 3/3 =====
  [Batch 200] Loss: 1.0702
  [Batch 400] Loss: 0.9003
  [Batch 600] Loss: 1.0390
Epoch 3 Train Loss: 1.0194 | Train Acc: 64.63%


In [6]:
# ============================================================
# 6. Evaluation on Test Set
# ============================================================

def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0
    class_correct = [0] * 10
    class_total = [0] * 10

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            for i in range(labels.size(0)):
                label = labels[i].item()
                class_total[label] += 1
                class_correct[label] += (predicted[i].item() == label)

    overall_acc = correct / total
    print(f"\nTest Accuracy: {overall_acc * 100:.2f}%")

    print("\nPer-class accuracy:")
    for i, cls in enumerate(classes):
        if class_total[i] > 0:
            acc = 100.0 * class_correct[i] / class_total[i]
            print(f"  {cls:5s}: {acc:5.2f}%")
        else:
            print(f"  {cls:5s}: N/A")

evaluate(model, test_loader)



Test Accuracy: 65.91%

Per-class accuracy:
  plane: 62.10%
  car  : 77.40%
  bird : 53.70%
  cat  : 39.70%
  deer : 60.00%
  dog  : 64.10%
  frog : 79.30%
  horse: 67.60%
  ship : 84.10%
  truck: 71.10%
